# Model C on Kaggle: SD1.5 + ControlNet + IP-Adapter

Settings: **GPU T4 x1** (or P100), **Internet on**. Attach the team's private data Kaggle Dataset, plus
the previous version of this notebook's output when resuming LoRA training.

Run the sections you need; each is independent once setup is done. Everything written under
`/kaggle/working` is kept in the notebook version's output.

In [ ]:
REPO_URL = "https://github.com/<org>/<repo>.git"   # fill in
BRANCH = "main"
!git clone -q -b $BRANCH $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Export evaluation PNGs (test split)
Needs the shared Dataset (`src.data.build_dataset`). Run once; reuse `data/eval` afterwards.

In [ ]:
!python -m scripts.export_eval_assets --split test --out data/eval

## 2. Tune on val
Coordinate search over `tune.grid` in `configs/controlnet.yaml`. Resumes from `results/tune_controlnet.csv` if this session is restarted with the previous output copied back.

In [ ]:
!python -m scripts.tune_controlnet --config configs/controlnet.yaml
!cat results/tune_controlnet_best.yaml

## 3. Test outputs
Generated by the shared `scripts/sample.py` with `ControlNetColorizer.from_config`, using the tuned `model:` section, so all three models are sampled by the same code. It writes `outputs/controlnet/<mode>/<id>_s<seed>.png`.

In [ ]:
# !python -m scripts.sample --config results/tune_controlnet_best.yaml --model controlnet --mode self_ref

## 4. Evaluate

In [ ]:
for mode in ["self_ref", "cross_ref", "shuffled_ref"]:
    !python -m scripts.evaluate --model controlnet --mode $mode

## 5. Optional LoRA (Experiment 4)
Stops cleanly before Kaggle's 12 h limit. In the next session, attach this notebook's output and pass `--init-from`.

In [ ]:
INIT = ""   # e.g. "--init-from /kaggle/input/<this-notebook>/repo/checkpoints/controlnet_lora"
!python -m scripts.train_lora --config configs/controlnet.yaml --max-minutes 660 $INIT
!tail -n 5 checkpoints/controlnet_lora/log.csv